In [8]:
print("sjnfdfG|")

sjnfdfG|


In [9]:
import os
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/sahilkumarrock8084/Wine-Quality-Data-Science-Projects-MLOPS-.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="sahilkumarrock8084"
os.environ["MLFLOW_TRACKING_PASSWORD"]="29bedcf08ad331f91cab5cb026e8f07c5075ebaa"


In [10]:
%pwd
import os
os.chdir("D:\mlops\Wine-Quality-Data-Science-Projects-MLOPS-")
%pwd


'D:\\mlops\\Wine-Quality-Data-Science-Projects-MLOPS-'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_path: Path
    model_path: Path
    all_params: dict
    metric_file_path: Path
    target_column: str
    mlflow_uri: str
    
    

In [12]:
from src.DATA_SCIENCE_Project.constants import *
from src.DATA_SCIENCE_Project.utils.helper import read_yaml,create_directories,save_json


class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config =  self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])
        
        model_eval_config  = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_path=config.test_path,
            model_path = config.model_path,
            all_params= params,
            metric_file_path= config.metric_file_path,
            target_column = schema.name,
            mlflow_uri="https://dagshub.com/sahilkumarrock8084/Wine-Quality-Data-Science-Projects-MLOPS-.mlflow"
        
            
        )
        return model_eval_config

In [13]:
import os
from src.DATA_SCIENCE_Project import logger
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
import mlflow
import mlflow.sklearn
import numpy as np
from urllib.parse import urlparse



class ModelEvaluationComponent:
        def __init__(self,config:ModelEvaluationConfig):
            self.config = config
            
        def eval_metrics(self,actual,pred):
            rmse = np.sqrt(mean_squared_error(actual,pred))
            mae = mean_absolute_error(actual,pred)
            r2 = r2_score(actual,pred)
            
            return rmse,mae,r2
        def log_into_mlflow(self):
            test_data = pd.read_csv(self.config.test_path)
            
            print(test_data.columns)
            
            model = joblib.load(self.config.model_path)
            print("Col: ",self.config.target_column)
            
            test_x = test_data.drop([self.config.target_column],axis=1)
            test_y = test_data[[self.config.target_column]]
            
            mlflow.set_registry_uri(self.config.mlflow_uri)
            tracking_uri_type_story = urlparse(mlflow.get_tracking_uri()).scheme
            
            with mlflow.start_run():
                
                prediction = model.predict(test_x)   
                
                (rmse,mae,r2) = self.eval_metrics(test_y,prediction) 
                scores = {'RMSE':rmse,'MAE':mae,'R2':r2} 
                
                save_json(path= Path(self.config.metric_file_path),data=scores) 
                
                mlflow.log_params(self.config.all_params)
                
                mlflow.log_metrics(scores)
                
                if tracking_uri_type_story != 'file':
                    
                    mlflow.sklearn.log_model(model,'model',registered_model_name="ElasticNet")
                else:
                    mlflow.sklearn.log_model(model,"model")
                    
            
            




In [14]:
try:
    cfm = ConfigurationManager()
    model_evaluate_manager = cfm.get_model_evaluation_config()
    model_evaluate_component = ModelEvaluationComponent(model_evaluate_manager)
    model_evaluate_component.log_into_mlflow()
    logger.info("Pipeline Ran Sucesssfully ✅😭 ")
except Exception as e:
    logger.error("Are bhia kuch Error Aaa Gaya hai Yaar....")
    logger.error(e)
    raise e

[25-04-2026 07:46:39 PM - helper - INFO - yaml file: config\config.yaml Loaded Sucessfully]
[25-04-2026 07:46:39 PM - helper - INFO - yaml file: params.yaml Loaded Sucessfully]
[25-04-2026 07:46:39 PM - helper - INFO - yaml file: schema.yaml Loaded Sucessfully]
[25-04-2026 07:46:39 PM - helper - INFO - Created Directory of path : artifacts]
[25-04-2026 07:46:39 PM - helper - INFO - Created Directory of path : artifacts/model_evaluation]
Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')
Col:  quality
[25-04-2026 07:46:39 PM - helper - INFO - Json File Saved to the path: artifacts\model_evaluation\metrics.json]


2026/04/25 19:46:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 19:46:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticNet'.
2026/04/25 19:46:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNet, version 1
Created version '1' of model 'ElasticNet'.


🏃 View run serious-finch-923 at: https://dagshub.com/sahilkumarrock8084/Wine-Quality-Data-Science-Projects-MLOPS-.mlflow/#/experiments/0/runs/e643b94f766c493282e9dd4ea624698b
🧪 View experiment at: https://dagshub.com/sahilkumarrock8084/Wine-Quality-Data-Science-Projects-MLOPS-.mlflow/#/experiments/0
[25-04-2026 07:47:01 PM - 3689569046 - INFO - Pipeline Ran Sucesssfully ✅😭 ]
